In [ ]:
from pydantic import BaseModel, PositiveInt, ValidationError
from typing import Tuple, Literal

class Column(BaseModel):
    name: str = ""
    shape: Tuple[PositiveInt, PositiveInt] = (0, 0)
    data_type: Literal["int", "float", "str", "bool"]

external_data_01 = {
    "name": "id",
    "shape": (1,1),
    "data_type": "str"
}  

external_data_02 = {
    "name": "id",
    "shape": (0,0),
    "data_type": "list"
}  

try:
    Column(**external_data_01)  
    Column(**external_data_02)  
except ValidationError as e:
    print(e.errors())


## Real case


https://amazonwebshark.com/python-data-validation-and-observability-as-code-with-pydantic/

In [ ]:
!pip install fastexcel

In [3]:
import polars as pl 

file_path: str = "/home/user/py-pannguyen-learn/data/raw/sales_data_sample.xlsx"
df = pl.read_excel(file_path)
df.head()

Order_ID,Date,Category,Region,Customer_Segment,Payment_Method,Quantity,Unit_Price,Discount_Rate,Customer_Rating,Returned,Subcategory,Total_Price,Discount_Amount,Net_Sales,Shipping_Cost,Profit_Margin,Profit,Month
str,date,str,str,str,str,i64,f64,f64,i64,bool,str,f64,f64,f64,f64,f64,f64,str
"""ORD-10000""",2024-07-12,"""Books""","""East""","""Business""","""PayPal""",1,316.58,0.2,5,false,"""Comics""",316.58,63.316,253.264,0.0,0.33,83.57712,"""2024-07"""
"""ORD-10001""",2025-03-15,"""Sports""","""West""","""Premium""","""Bank Transfer""",5,279.76,0.1,null,true,"""Water Sports""",1398.8,139.88,1258.92,0.0,0.2,251.784,"""2025-03"""
"""ORD-10002""",2024-12-27,"""Home & Kitchen""","""Central""","""Business""","""Bank Transfer""",6,209.61,0.05,4,false,"""Furniture""",1257.66,62.883,1194.777,0.0,0.37,442.06749,"""2024-12"""
"""ORD-10003""",2024-07-16,"""Home & Kitchen""","""North""","""Regular""","""Credit Card""",3,265.47,0.15,4,false,"""Kitchenware""",796.41,119.4615,676.9485,0.0,0.23,155.698155,"""2024-07"""
"""ORD-10004""",2024-06-11,"""Electronics""","""North""","""New""","""Bank Transfer""",9,449.32,0.0,4,false,"""Accessories""",4043.88,0.0,4043.88,0.0,0.37,1496.2356,"""2024-06"""


In [6]:
origin_cols: list = df.columns

In [13]:
def normailzed_columns(columns: pl.DataFrame.columns) -> list:
    return [col.strip().lower() for col in columns]
normalized_cols: list = normailzed_columns(origin_cols)
normalized_cols

['order_id',
 'date',
 'category',
 'region',
 'customer_segment',
 'payment_method',
 'quantity',
 'unit_price',
 'discount_rate',
 'customer_rating',
 'returned',
 'subcategory',
 'total_price',
 'discount_amount',
 'net_sales',
 'shipping_cost',
 'profit_margin',
 'profit',
 'month']

In [14]:
df.columns = normalized_cols
df.head()

order_id,date,category,region,customer_segment,payment_method,quantity,unit_price,discount_rate,customer_rating,returned,subcategory,total_price,discount_amount,net_sales,shipping_cost,profit_margin,profit,month
str,date,str,str,str,str,i64,f64,f64,i64,bool,str,f64,f64,f64,f64,f64,f64,str
"""ORD-10000""",2024-07-12,"""Books""","""East""","""Business""","""PayPal""",1,316.58,0.2,5,false,"""Comics""",316.58,63.316,253.264,0.0,0.33,83.57712,"""2024-07"""
"""ORD-10001""",2025-03-15,"""Sports""","""West""","""Premium""","""Bank Transfer""",5,279.76,0.1,null,true,"""Water Sports""",1398.8,139.88,1258.92,0.0,0.2,251.784,"""2025-03"""
"""ORD-10002""",2024-12-27,"""Home & Kitchen""","""Central""","""Business""","""Bank Transfer""",6,209.61,0.05,4,false,"""Furniture""",1257.66,62.883,1194.777,0.0,0.37,442.06749,"""2024-12"""
"""ORD-10003""",2024-07-16,"""Home & Kitchen""","""North""","""Regular""","""Credit Card""",3,265.47,0.15,4,false,"""Kitchenware""",796.41,119.4615,676.9485,0.0,0.23,155.698155,"""2024-07"""
"""ORD-10004""",2024-06-11,"""Electronics""","""North""","""New""","""Bank Transfer""",9,449.32,0.0,4,false,"""Accessories""",4043.88,0.0,4043.88,0.0,0.37,1496.2356,"""2024-06"""


In [39]:
from pydantic import BaseModel, Field, ValidationError, field_validator
import datetime


class SaleDataModel(BaseModel):
    order_id: str = Field(
        min_length=1,
        max_length=50,
        validate_default=True,
        strict=True,
        repr=True
    )
    discount_rate: float = Field(
        ge=0,
        le=1,
        validate_default=True,
        strict=True,
        repr=True
    )
    month: str = Field(
        ge=datetime.date(2022, 1, 1).strftime("%Y-%m"),
        le=datetime.datetime.now().strftime("%Y-%m"),
        validate_default=True,
        strict=False,
        repr=True
    )

    @field_validator("order_id",  mode="before")
    @classmethod
    def validate_not_empty(cls, value, info):
        """Validate that a string field is not empty."""
        if str(value).strip() == "":
            raise ValueError(f"{info.field_name} must not be null, na or empty")
        return value

In [63]:
df_with_index = df.with_row_index()
index = 1
for row in df_with_index.iter_rows(named=True):
    try:
        validate_result = SaleDataModel(**row)
    except  (ValidationError, ValueError, Exception) as e:
        print(index, str(e))
    finally:
        index +=1

11 1 validation error for SaleDataModel
discount_rate
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/float_type
102 1 validation error for SaleDataModel
discount_rate
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/float_type
120 1 validation error for SaleDataModel
discount_rate
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/float_type
135 1 validation error for SaleDataModel
discount_rate
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/float_type
138 1 validation error for SaleDataModel
discount_rate
  Input should be a valid number [type=float_t

In [ ]:
from enum import Enum

class HTTPMethod(Enum):
    POST = "POST"
    GET = "GET"

HTTPMethod.POST.value

In [ ]:
from enum import StrEnum

class Method(StrEnum):
    POST = "POST"

Method.POST.value